# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

#### My lane: Lane 2 — Refresh / Content Opportunity Scoring

- I chose this lane because the starter pipeline already demonstrates a concrete, measurable win: the Random Forest model achieves 0.740 Precision@50 versus 0.240 for the hand-written baseline — a 3x improvement. As an ML Engineer, I want to test whether this result holds on the full warehouse dataset (78.8M rows) and build a production-ready ranking system that content teams can actually use. This lane has clear business value: helping reviewers spend their limited time on the pages most likely to need attention.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

- What decision does your work improve?
Which content pages should a content reviewer inspect first to maximize the chance of catching declining or underperforming pages.

- Who acts on it?
A content reviewer or content operations team opens a ranked list, starts at the top, and reviews/refreshes pages in order of priority.

- What does a wrong recommendation cost?

<small>

**What decision does your work improve?**
> Which content pages should a content reviewer inspect first to maximize the chance of catching declining or underperforming pages.

**Who acts on it?**
> A content reviewer or content operations team opens a ranked list, starts at the top, and reviews/refreshes pages in order of priority.

**What does a wrong recommendation cost?**

| Error type | What it means | Cost |
|------------|---------------|------|
| **False Positive** | Reviewer wastes time on a page that doesn't need work | Wasted human hours; opportunity cost — they could have reviewed a real problem instead |
| **False Negative** | A declining page goes unnoticed | Lost traffic, lost revenue, competitive disadvantage over time |

</small>

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:
import os

print(os.getcwd())
print(os.listdir())

g:\internship\FlyRank-AI-Internship\work\notebooks
['capstone.ipynb', 'w01_research_question.ipynb', 'w02_ml_task_framing.ipynb', 'w03_data_contract.ipynb', 'w03_feature_leakage_check.ipynb', 'w04_baseline_score.ipynb', 'w04_signal_audit.ipynb', 'w05_model.ipynb', 'w06_validation_audit.ipynb', 'w07_action_playbook.ipynb']


In [10]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f" Total pages in starter dataset: {len(df):,}")
print("-" * 50)

# 1. Label distribution — shows this is a real problem worth solving
declining = df[df['trend_direction'] == 'down'].shape[0]
declining_pct = declining / len(df) * 100
print(f" Declining pages: {declining:,} ({declining_pct:.1f}%)")
print(f" Non-declining pages: {len(df) - declining:,} ({100 - declining_pct:.1f}%)")
print("-" * 50)

# 2. Pages with enough demand to matter (filters out noise)
high_demand = df[df['impressions_90d'] >= 500].shape[0]
print(f" Pages with >=500 impressions (high demand): {high_demand:,} ({high_demand/len(df)*100:.1f}%)")
print(f"   → These are the pages worth reviewing; the rest may be too low-volume to act on")
print("-" * 50)

# 3. Starter model results — the key proof that ML adds value
print(" Starter model Precision@50 (top 50 pages):")
print(f"   • Baseline rules: 0.240 → {int(0.240 * 50)} correct out of 50")
print(f"   • Random Forest: 0.740 → {int(0.740 * 50)} correct out of 50")
print(f"   → That's ~3x more true positives found in the top 50")
print("-" * 50)

# Show that ML finds more true positives earlier
print(" Why this matters:")
print("   A reviewer checking 50 pages with Random Forest finds")
print(f"   {int(0.740 * 50) - int(0.240 * 50)} more real problems than using the baseline rule.")
print("   That's the business value of ML in this context.")

 Total pages in starter dataset: 30,000
--------------------------------------------------
 Declining pages: 16,262 (54.2%)
 Non-declining pages: 13,738 (45.8%)
--------------------------------------------------
 Pages with >=500 impressions (high demand): 16,726 (55.8%)
   → These are the pages worth reviewing; the rest may be too low-volume to act on
--------------------------------------------------
 Starter model Precision@50 (top 50 pages):
   • Baseline rules: 0.240 → 12 correct out of 50
   • Random Forest: 0.740 → 37 correct out of 50
   → That's ~3x more true positives found in the top 50
--------------------------------------------------
 Why this matters:
   A reviewer checking 50 pages with Random Forest finds
   25 more real problems than using the baseline rule.
   That's the business value of ML in this context.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

<font size="2">

### What I CAN claim:

| Claim | Why safe |
|-------|----------|
| "I can rank pages by their likelihood of being declining, based on observable historical signals" | The model uses only signals available at prediction time (impressions, clicks, position, engagement, age, etc.) |
| "This ranking outperforms a fixed heuristic baseline on the starter dataset" | We measured Precision@50 on client-holdout validation; the numbers are in the output |
| "The model identifies true positives earlier in the queue than the baseline" | Precision@50 is higher, which means more true positives in the top 50 |
| "This is a decision-support tool — it helps reviewers prioritize, but does not replace human judgment" | The output is a ranked queue with reason codes; a human makes the final call |

### What I CAN'T claim:

| Claim | Why NOT safe |
|-------|---------------|
| "Refreshing these pages WILL cause them to recover" | That would require a causal experiment or A/B test — this is observational data only |
| "I discovered Google's ranking factors" | I only looked at signals correlated with decline; correlation ≠ causation |
| "This model works on every client without adjustment" | Client-holdout validation helps, but real performance varies by client |
| "The model 'understands' content or meaning" | It only uses structured signals (counts, rates, tiers) — no text understanding |

### My honest framing:

> This is an **observational ranking system**. It identifies patterns that historically preceded decline based on observable signals (impressions, clicks, position, engagement, age, and derived metrics). The output is a prioritized review queue, not a guarantee of recovery. Any final decision about whether to refresh a page should involve human judgment, seasonal context, and business priorities.

### What I'm NOT doing:

| What I'm NOT doing | Why |
|-------------------|-----|
| Training a binary classifier on AI-session data | AI sessions are too sparse (30,177 rows vs 78.8M daily rows) |
| Rebuilding product flags and feeding them as features | That would create a circular result; I use only observable signals |
| Claiming causation | All claims are "observed" and "associated with" — never "causes" |
| Publishing sensitive data | No client names, domains, URLs, queries, or titles leave this notebook |

</font>

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.